# RF Board — DAC Test Notebook

f_DACCLK = 1 GHz

f_DATACLK = 125 MHz **CHECK IN XSA NAME** (no name-dataclk 125MHz)

f_OSTR = 15.625 MHz

f_SYNC = 15.625 MHz

f_DDS_CLK = (f_DATACLK)/2 

---
# **Load overlay:**

In [7]:
# Step 1 — Load the PL bitstream overlay
from pynq import Overlay
import time
ol = Overlay('./ADC_DAC12ST_DATACLK125MHz.xsa')
print('Overlay loaded.')

# Step 2 — Instantiate controller (no BRAM, but with AXI for DDS)
from pynq_controller import PynqController
BOARD_VER = 2
zynq = PynqController(ol, board_ver=BOARD_VER, enable_bram=False, enable_axi=True)
zynq.DACCLK_HZ = 1_000_000_000   # 1 GHz DAC clock

Overlay loaded.

[PynqController] Initializing (board v2)...
CDCE62005 ready (board v2): CLK=MIO12, MOSI=MIO10, MISO=MIO11, LE=MIO15
  CS_DAC1=MIO13, CS_DAC2=MIO14, CS_ADC=MIO0 — all deasserted
SPI3WireBus ready (board v2): CLK=MIO12, SDIO=MIO11
DAC3484 DAC1 ready (CS_DAC1)
DAC3484 DAC2 ready (CS_DAC2)
AD9643 ADC ready
  (BRAM controller disabled)
    ⚠ DDS_CTRL GPIO not found in overlay (ST_0/DDS_CTRL). Verify XSA has DDS_CTRL AXI GPIO block.
  (AXI subsystem initialized for DDS)
✓ PynqController ready



---
# **According to power-up sequence in datasheet**
### 1. Check DAC not in sleep (DAC1: P4 2-3, DAC2: P6 2-3) 
### 2. Set TXENABLE low (DAC1: P3 2-3, DAC2: P5 2-3) 
### 3. Turn on power supply (5.5v 2A)

In [8]:
### Supply DACCLK and OSTR
# DACCLK1 DACCLK2 1 GHz, OSTR effective 15.625 MHz, FPGA_CLK ADC_CLK off
CDCE_CONFIG_no_FPGA = '8340032083400321EA040302811C0303EA84031410000BE504BE09F6BD0037F7'
zynq.configure_clock(CDCE_CONFIG_no_FPGA,debug=1)

Configuring CDCE62005...
  reg[0] <- 0x8340032
  reg[1] <- 0x8340032
  reg[2] <- 0xEA04030
  reg[3] <- 0x811C030
  reg[4] <- 0xEA84031
  reg[5] <- 0x10000BE
  reg[6] <- 0x04BE09F
  reg[7] <- 0xBD0037F
  reg[8] = 0x80009BF (OK)
Done.


True

### **Toggle RESETB**

In [9]:
## Configure DAC input
DDS_CLK_HZ = 62_500_000
from pynq_dds import DAC2_DDS
dds2 = DAC2_DDS(ol, dds_clk_hz=DDS_CLK_HZ)

# Configurable frequency and phase
AB_FREQ = 10.0    # MHz
AB_PHASE = 0.0   # degrees
CD_FREQ = 10.0    # MHz
CD_PHASE = 0.0   # degreesprint('Overlay loaded.')# ___________________________________________________________________________________________________________________
#--------------------------------------------- DAC2 Initialization- Part 1 ------------------------------------------
# ___________________________________________________________________________________________________________________
### **Program SIF registers:**
zynq.dac2.write_register(0x00, 0x089F) # INTERP = 16, fifo on, clkdiv_sync_ena invsincAB+CD
zynq.dac2.write_register(0x01, 0x050E)
zynq.dac2.write_register(0x02, 0xF052) # 16bit_in, mixer_ena, nco_ena, twos_comp
zynq.dac2.write_register(0x03, 0xF000)
zynq.dac2.write_register(0x07, 0xD8FF) # un-mask FIFO collison, DACCLK-gone, DATACLK-gone

# # registers 0x08-0x11 are QMC module, unused
# # registers 0x12-0x13 NCO phase, unused 

# set_dac_nco(self, channel: int, freq_mhz: float, phase_deg: float = 0)
zynq.dac2.set_dac_nco(1, 199.0, 0)
zynq.dac2.set_dac_nco(2, 199.0, 0)

# PLL Disable:
zynq.dac2.write_register(0x18, 0x0000) # pll_ena=0, 
zynq.dac2.write_register(0x19, 0x0000)
zynq.dac2.write_register(0x1A, 0x0020) # pll_sleep=1

# # PLL Enable:
# zynq.dac2.write_register(0x18, 0x0460)  # pll_ena=1, pll_cp[1:0]=01 single charge pump, pll_p[2:0]=100 (4)
# zynq.dac2.write_register(0x19, 0x0434)  # pll_m[7:0]= 0 000 0100 (4), pll_n[3:0]=0011 (4), pll_vcoitune[1:0]=01
# zynq.dac2.write_register(0x1A, 0xFC00)  # pll_vco[5:0]=1111 11 (4GHz), pll_sleep=0
# lfvolt2 = zynq.dac2.read_register(0x18) # read pll_lfvolt to check for lock
# print(f" DAC2- PLL LOCK : REG0x18 = 0x{lfvolt2:04X}")

zynq.dac2.write_register(0x1B, 0x0800) # fuse_sleep=1

zynq.dac2.write_register(0x1E, 0x8888) # sync_sel for QMC module- SIF_SYNC
zynq.dac2.write_register(0x1F, 0x2224) # syncsel_mixerAB, syncsel_mixerCD, syncsel_nco: OSTR, syncsel_dataformatter: SYNC 
zynq.dac2.write_register(0x20, 0x1400) # syncsel_fifoin: SYNC, syncsel_fifoout: OSTR, clkdiv_sync_sel: OSTR


# ___________________________________________________________________________________________________________________
# xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx DAC1 Initialization- Part 1 xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx
# ___________________________________________________________________________________________________________________

zynq.dac1.write_register(0x00, 0x089F) # INTERP = 16, fifo on, clkdiv_sync_ena invsincAB+CD
zynq.dac1.write_register(0x01, 0x050E)
zynq.dac1.write_register(0x02, 0xF052) # 16bit_in, mixer_ena, nco_ena, twos_comp
zynq.dac1.write_register(0x03, 0xF000)
zynq.dac1.write_register(0x07, 0xD8FF) # un-mask FIFO collison, DACCLK-gone, DATACLK-gone
time.sleep(0.1)

# # registers 0x08-0x11 are QMC module, unused
# # registers 0x12-0x13 NCO phase, unused 

zynq.dac1.set_dac_nco(1, 156.0, 0)
zynq.dac1.set_dac_nco(2, 156.0, 0)

# PLL Disable:
zynq.dac1.write_register(0x18, 0x0000) # pll_ena=0, 
zynq.dac1.write_register(0x19, 0x0000)
zynq.dac1.write_register(0x1A, 0x0020) # pll_sleep=1

# # PLL Enable:
# zynq.dac1.write_register(0x18, 0x0460)  # pll_ena=1, pll_cp[1:0]=01 single charge pump, pll_p[2:0]=100 (4)
# zynq.dac1.write_register(0x19, 0x0434)  # pll_m[7:0]= 0 000 0100 (4), pll_n[3:0]=0011 (4), pll_vcoitune[1:0]=01
# zynq.dac1.write_register(0x1A, 0xFC00)  # pll_vco[5:0]=1111 11 (4GHz), pll_sleep=0
# lfvolt2 = zynq.dac1.read_register(0x18) # read pll_lfvolt to check for lock
# print(f"DAC1- PLL LOCK: REG0x18 = 0x{lfvolt2:04X}")

zynq.dac1.write_register(0x1B, 0x0800) # fuse_sleep=1

zynq.dac1.write_register(0x1E, 0x8888) # sync_sel for QMC module- SIF_SYNC
zynq.dac1.write_register(0x1F, 0x2224) # syncsel_mixerAB, syncsel_mixerCD, syncsel_nco: OSTR, syncsel_dataformatter: SYNC 
zynq.dac1.write_register(0x20, 0x1400) # syncsel_fifoin: SYNC, syncsel_fifoout: OSTR, clkdiv_sync_sel: OSTR


# ___________________________________________________________________________________________________________________
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~ Clock configuration for both DACs ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# ___________________________________________________________________________________________________________________
## Add FPGACLK in order to start all LVDS data and clocks
# DACCLK1 DACCLK2 1 GHz, OSTR effective 15.625 MHz, FPGA_CLK 250 MHz, ADC_CLK off
CDCE_CONFIG_wFPGA = '834003208340032181040302811C0303EB84031410000BE504BE09F6BD0037F7'
zynq.configure_clock(CDCE_CONFIG_wFPGA)
print('CDCE configured with FPGACLK')
time.sleep(0.001)

# ___________________________________________________________________________________________________________________
#--------------------------------------------- DAC2 Initialization- Part 2 -------------------------------------------
# ___________________________________________________________________________________________________________________
## Check pll lock and reset alarms register
lfvolt2 = zynq.dac2.read_register(0x18) # read pll_lfvolt to check for lock
print(f"✓ PLL LOCK: REG0x18 = 0x{lfvolt2:04X}")

zynq.dac2.write_register(0x05, 0x0000) # turn of Alarms for check
time.sleep(0.001)
reg5_DAC2 = zynq.dac2.read_register(0x05) # read reg5 to check for alarms
print(f"DAC2- Alarms Register: REG0x05 = 0x{reg5_DAC2:04X}")
time.sleep(0.001)

### Disable clock divider sync
zynq.dac2.write_register(0x1F, 0x2224) # turn on sif_sync
zynq.dac2.write_register(0x00, 0x089B) # INTERP = 16, fifo on, clkdiv_sync_ena=0, invsincAB+CD=1
time.sleep(0.001)
zynq.dac2.write_register(0x1F, 0x2228) # turn off sif_sync, syncsel_dataformatter- no sync
zynq.dac2.write_register(0x20, 0x0000) # Disable FIFO input and output pointer sync
zynq.dac2.write_register(0x03, 0xF001) # Set TXENABLE high
zynq.dac2.write_register(0x05, 0x0000) # turn of Alarms for check
time.sleep(0.001)
reg5_DAC2 = zynq.dac2.read_register(0x05) # read reg5 to check for alarms
print(f"✓ Alarms Register : REG0x05 = 0x{reg5_DAC2:04X}")
zynq.dac2.write_register(0x24, 0x1C00)  


# ___________________________________________________________________________________________________________________
# xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx DAC1 Initialization- Part 2 xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx
# ___________________________________________________________________________________________________________________

## Check pll lock and reset alarms register
lfvolt1 = zynq.dac1.read_register(0x18) # read pll_lfvolt to check for lock
print(f"DAC1- PLL LOCK: REG0x18 = 0x{lfvolt1:04X}")

zynq.dac1.write_register(0x05, 0x0000) # turn of Alarms for check
time.sleep(0.001)
reg5_DAC1 = zynq.dac1.read_register(0x05) # read reg5 to check for alarms
print(f"DAC1- Alarms Register: REG0x05 = 0x{reg5_DAC1:04X}")
time.sleep(0.001)

### Disable clock divider sync
zynq.dac1.write_register(0x1F, 0x2224) # turn on sif_sync
zynq.dac1.write_register(0x00, 0x089B) # INTERP = 16, fifo on, clkdiv_sync_ena=0, invsincAB+CD=1
time.sleep(0.001)
zynq.dac1.write_register(0x1F, 0x2228) # turn off sif_sync, syncsel_dataformatter- no sync


zynq.dac1.write_register(0x20, 0x0000) # Disable FIFO input and output pointer sync
zynq.dac1.write_register(0x03, 0xF001) # Set TXENABLE high
zynq.dac1.write_register(0x05, 0x0000) # turn of Alarms for check
time.sleep(0.001)
reg5_DAC1 = zynq.dac1.read_register(0x05) # read reg5 to check for alarms
print(f"DAC1- Alarms Register: REG0x05 = 0x{reg5_DAC1:04X}")

zynq.dac1.write_register(0x24, 0x1C00)  

dds2.set(ab_freq_mhz=AB_FREQ,  ab_phase_deg=AB_PHASE,
         cd_freq_mhz=CD_FREQ,  cd_phase_deg=CD_PHASE)

print(f"\n✓ DAC2 DDS configured ({DDS_CLK_HZ/1e6:.1f} MHz clock):")
print(f"  AB: {AB_FREQ} MHz @ {AB_PHASE}°")
print(f"  CD: {CD_FREQ} MHz @ {CD_PHASE}°")

DAC2 DDS ready  (CLK = 62.5 MHz)
DAC2 NCO configured: word=0x32F1A9FB
DAC2 NCO configured: word=0x32F1A9FB
DAC1 NCO configured: word=0x27EF9DB2
DAC1 NCO configured: word=0x27EF9DB2
CDCE configured with FPGACLK
✓ PLL LOCK: REG0x18 = 0x0007
DAC2- Alarms Register: REG0x05 = 0x3968
✓ Alarms Register : REG0x05 = 0x1878
DAC1- PLL LOCK: REG0x18 = 0x0007
DAC1- Alarms Register: REG0x05 = 0x3978
DAC1- Alarms Register: REG0x05 = 0x1878
AB: 10.000 MHz  0.0°  PINC=0x28F5C28F  POFF=0x00000000
CD: 10.000 MHz  0.0°  PINC=0x28F5C28F  POFF=0x00000000

✓ DAC2 DDS configured (62.5 MHz clock):
  AB: 10.0 MHz @ 0.0°
  CD: 10.0 MHz @ 0.0°


In [4]:
## Configure DAC input
DDS_CLK_HZ = 62_500_000
from pynq_dds import DAC2_DDS
dds2 = DAC2_DDS(ol, dds_clk_hz=DDS_CLK_HZ)

# Configurable frequency and phase
AB_FREQ = 5.0    # MHz
AB_PHASE = 0.0   # degrees
CD_FREQ = 5.0    # MHz
CD_PHASE = 0.0   # degrees

dds2.set(ab_freq_mhz=AB_FREQ,  ab_phase_deg=AB_PHASE,
         cd_freq_mhz=CD_FREQ,  cd_phase_deg=CD_PHASE)

print(f"\n✓ DAC2 DDS configured ({DDS_CLK_HZ/1e6:.1f} MHz clock):")
print(f"  AB: {AB_FREQ} MHz @ {AB_PHASE}°")
print(f"  CD: {CD_FREQ} MHz @ {CD_PHASE}°")

DAC2 DDS ready  (CLK = 62.5 MHz)
AB: 5.000 MHz  0.0°  PINC=0x147AE147  POFF=0x00000000
CD: 5.000 MHz  0.0°  PINC=0x147AE147  POFF=0x00000000

✓ DAC2 DDS configured (62.5 MHz clock):
  AB: 5.0 MHz @ 0.0°
  CD: 5.0 MHz @ 0.0°


In [6]:
## Turn off DAC input
dds2.off()

AB: 0.000 MHz  0.0°  PINC=0x00000000  POFF=0x00000000
CD: 0.000 MHz  0.0°  PINC=0x00000000  POFF=0x00000000
DAC2 DDS off


In [10]:
zynq.read_all_dac_registers(dac_num=1)


[DAC1] Reading all registers:
  0x00: 0x089B
  0x01: 0x050E
  0x02: 0xF052
  0x03: 0xF001
  0x04: 0xB7D6
  0x05: 0x0878
  0x06: 0x3900
  0x07: 0xD8FF
  0x08: 0x0000
  0x09: 0x8000
  0x0A: 0x0000
  0x0B: 0x0000
  0x0C: 0x0400
  0x0D: 0x0400
  0x0E: 0x0400
  0x0F: 0x0400
  0x10: 0x0000
  0x11: 0x0000
  0x12: 0x0000
  0x13: 0x0000
  0x14: 0x0000
  0x15: 0x0000
  0x16: 0x0000
  0x17: 0x0000
  0x18: 0x0007
  0x19: 0x0000
  0x1A: 0x0020
  0x1B: 0x0800
  0x1C: 0x0000
  0x1D: 0x0000
  0x1E: 0x8888
  0x1F: 0x2228
  0x20: 0x0000
  0x21: 0x0000
  0x22: 0x1B1B
  0x23: 0xFFFF
  0x24: 0x1C00
  0x25: 0x7A7A
  0x26: 0xB6B6
  0x27: 0xEAEA
  0x28: 0x4545
  0x29: 0x1A1A
  0x2A: 0x1616
  0x2B: 0xAAAA
  0x2C: 0xC6C6
  0x2D: 0x0004
  0x2E: 0x0000
  0x2F: 0x0000
  0x30: 0x0000
  0x7F: 0x540C


{0: 2203,
 1: 1294,
 2: 61522,
 3: 61441,
 4: 47062,
 5: 2168,
 6: 14592,
 7: 55551,
 8: 0,
 9: 32768,
 10: 0,
 11: 0,
 12: 1024,
 13: 1024,
 14: 1024,
 15: 1024,
 16: 0,
 17: 0,
 18: 0,
 19: 0,
 20: 0,
 21: 0,
 22: 0,
 23: 0,
 24: 7,
 25: 0,
 26: 32,
 27: 2048,
 28: 0,
 29: 0,
 30: 34952,
 31: 8744,
 32: 0,
 33: 0,
 34: 6939,
 35: 65535,
 36: 7168,
 37: 31354,
 38: 46774,
 39: 60138,
 40: 17733,
 41: 6682,
 42: 5654,
 43: 43690,
 44: 50886,
 45: 4,
 46: 0,
 47: 0,
 48: 0,
 127: 21516}